In [ ]:
"""Data construction: merge FRED macro series + the provided shock into one"""
# === quarterly dataset used forr both Part A (Local Projections) and Part B (SVAR).
import pandas as pd      # use for ndata loading, date handling, merging
import numpy as np       # use for natural logs

In [ ]:
"""Three of the four macro series downloaded from FRED are in MONTHLY frequency, but the analysis is quarterly. We collapse months to quarters by AVERAGING the three
months in each quarter (the standard convention for rates / price indices)"""

"""Function to calculate quarterly mean value"""

def quarterly_mean(file_path, value_col, new_col):
    """Read a monthly FRED CSV and return a quarterly-average Series.
    file_path : CSV with columns [observation_date, <value_col>]
    value_col : the FRED column name to read (e.g. 'FEDFUNDS')
    new_col   : name of the output Series (e.g. 'ffr')"""

    s = pd.read_csv(file_path, parse_dates=["observation_date"]).set_index("observation_date")[value_col]
    s.index = s.index.to_period("Q")
    return s.groupby(level = 0).mean().rename(new_col)  

# Apply to the three monthly series 
cpi_q = quarterly_mean("CPIAUCSL.csv", "CPIAUCSL", "cpi")
ffr_q = quarterly_mean("FEDFUNDS.csv","FEDFUNDS","ffr")
unrate_q = quarterly_mean("UNRATE.csv","UNRATE","unrate")

print(cpi_q.head())
print(ffr_q.head())
print(unrate_q.head())

In [ ]:
"""Real GDP (GDPC1) is ALREADY quarterly on FRED, so we do not run the function for calculating quarterly average for it we just read it and label its index by quarter."""
gdp_s = pd.read_csv("GDPC1.csv", parse_dates=["observation_date"]).set_index("observation_date")["GDPC1"]
gdp_s.index = gdp_s.index.to_period("Q")
gdp_q = gdp_s.rename("gdp")

"""Take natural logs of GDP and CPI. Logs because: (i) 100 x a log change ~ a percent change, so IRFs read directly in percent; (ii) it linearises exponential growth.
FFR and unrate are already in percent, so they are NOT logged."""
log_real_gdp = np.log(gdp_q).rename("log_real_gdp")
log_cpi      = np.log(cpi_q).rename("log_cpi")

print(log_real_gdp.head()); print(log_cpi.head())

In [ ]:
"""The provided shock file ships with columns [year, quarter, mp_shock] not in the 'YYYYQN' string format as described in the PDF.""" 
""" We build a quarterly index from the year/quarter integer columns so it aligns with the macro series used above."""
sh = pd.read_csv("shocks.csv")
sh_index = pd.PeriodIndex.from_fields(year=sh["year"], quarter=sh["quarter"], freq="Q")
mp_shock = pd.Series(sh["mp_shock"].values, index=sh_index, name="mp_shock")

"""Note: the shock data is only till 2007Q4 even though macro data runs to 2019Q4  this is the  constraint on the estimation sample, see the tail below)."""
print(mp_shock.head())
print(mp_shock.tail())

In [ ]:
""" Putting the five Series side-by-side. using pandas we align rows by matching quarter labels"""
data = pd.concat([log_real_gdp, log_cpi, ffr_q, unrate_q, mp_shock], axis=1)

"""Staying to the requested macro window 1969Q1–2019Q4. mp_shock will be NaN after 2007Q4; yet we KEEP those rows so Part A's leads y(t+h) can still
use post-2007 macro data. Each method forms its own valid sample as it moves forward."""
window = pd.period_range("1969Q1", "2019Q4", freq="Q")
data = data.reindex(window)
data.index.name = "quarter"

In [ ]:
"""Separately printing the missing row: only mp_shock has missing rows, 48 rows post-2007Q4 quarters."""

print("Rows total:", len(data))
print("Missing per column:\n", data.isna().sum())
print("\nmp_shock available:", data["mp_shock"].first_valid_index(),
      "to", data["mp_shock"].last_valid_index())

In [ ]:
""" Save the dataset, this will be used for Part A and B for analysis"""
data.to_csv("merged_data.csv")
print("Saved merged_data.csv")